# Đồ án 2 - Phần 2: Hồi quy tuyến tính đa biến với bộ dữ liệu Bike Sharing
### Nhóm 13

---

## 1. Tiền xử lý dữ liệu và EDA (Người 1)

### 1.1. Thống kê mô tả và trực quan hóa (EDA)
- Vẽ biểu đồ phân phối (Histogram) của các biến liên tục.
- Vẽ biểu đồ hộp (Boxplot) phát hiện ngoại lệ (outlier).
- Vẽ ma trận tương quan (Heatmap) giữa các biến.
- Phân tích phân phối của biến mục tiêu `cnt`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Đảm bảo import được các module trong thư mục
sys.path.append(os.path.abspath('.'))

# Đọc dữ liệu gốc có missing
df_raw = pd.read_csv('data/day.csv')
df_raw.head()

### 1.2. Xử lý Missing Values
- Cơ chế khuyết dữ liệu: Missing Completely At Random (MCAR).
- Áp dụng và so sánh các phương pháp điền khuyết (Mean, Median, KNN/Regression Imputation).
- So sánh phân phối trước và sau khi điền khuyết.

In [ ]:
# Code xử lý khuyết dữ liệu của Người 1

### 1.3. Gọi DataPipeline hoàn chỉnh

In [ ]:
from part2.data_pipeline import load_data
X_train, X_test, y_train, y_test, feature_names = load_data()
print("Kích thước tập huấn luyện:", X_train.shape)
print("Kích thước tập kiểm thử:", X_test.shape)
print("Đặc trưng:", feature_names)

## 2. Mô tả dữ liệu & Mô hình OLS (Người 2)

### 2.1. Mô tả bộ dữ liệu
- Trình bày nguồn gốc bộ dữ liệu.
- Trình bày ý nghĩa của các biến trong bộ dữ liệu.
- Lý giải sự phù hợp của bộ dữ liệu đối với bài toán hồi quy.

### 2.2. Huấn luyện mô hình OLS Baseline
- Chạy hồi quy tuyến tính trên tất cả các đặc trưng ban đầu.

In [ ]:
from part2.model_comparison import run_ols_baseline, compute_metrics
beta_baseline, mae_base, rmse_base, r2_base = run_ols_baseline(X_train, X_test, y_train, y_test, feature_names)

### 2.3. Lựa chọn biến (OLS Selection)
- Phân tích đa cộng tuyến (VIF) và giá trị p-value.
- Loại bỏ các biến không có ý nghĩa thống kê hoặc đa cộng tuyến cao.
- So sánh hiệu năng mô hình sau khi chọn biến.

In [ ]:
from part2.model_comparison import run_ols_selection
# run_ols_selection(X_train, X_test, y_train, y_test, feature_names)

### 2.4. Phân tích phần dư (Residual Analysis)
- Vẽ 4 biểu đồ chẩn đoán phần dư trên mô hình OLS tốt nhất.
- Đưa ra nhận xét chi tiết về các giả định của mô hình.

In [ ]:
from part2.model_comparison import plot_residuals
plot_residuals(X_train, y_train, beta_baseline)

## 3. Co rút hệ số - Ridge và Lasso (Người 3)

### 3.1. Hồi quy Ridge
- Sử dụng Cross-Validation để tìm tham số $\lambda$ tốt nhất.
- Vẽ đồ thị CV score (MSE) ứng với các giá trị $\lambda$ khác nhau.
- Trình bày kết quả đánh giá trên tập kiểm thử.

In [ ]:
from part2.model_comparison import run_ridge
beta_ridge, best_lam_ridge, mae_r, rmse_r, r2_r = run_ridge(X_train, X_test, y_train, y_test, feature_names)

### 3.2. Hồi quy Lasso
- Sử dụng Cross-Validation để tìm tham số $\lambda$ tốt nhất.
- Nhận xét các đặc trưng có hệ số bị triệt tiêu về 0.
- Trình bày kết quả đánh giá trên tập kiểm thử.

In [ ]:
from part2.model_comparison import run_lasso
beta_lasso, best_lam_lasso, mae_l, rmse_l, r2_l = run_lasso(X_train, X_test, y_train, y_test, feature_names)

### 3.3. So sánh hiệu năng các mô hình và kết luận

In [ ]:
from part2.model_comparison import print_comparison_table
print_comparison_table(
    ols_res=(mae_base, rmse_base, r2_base),
    ols_select_res=None,  # Cập nhật từ Người 2
    ridge_res=(mae_r, rmse_r, r2_r),
    lasso_res=(mae_l, rmse_l, r2_l)
)

## 4. Phương pháp nâng cao (Người 4)

### 4.1. Kernel Ridge Regression (RBF Kernel)

In [ ]:
from part2.advanced_methods import kernel_ridge_predict
y_pred_kr = kernel_ridge_predict(X_train, y_train, X_test, lam=1.0, length_scale=1.0)
mae_kr, rmse_kr, r2_kr = compute_metrics(y_test, y_pred_kr)
print(f"Kernel Ridge Regression -> MAE: {mae_kr:.4f}, RMSE: {rmse_kr:.4f}, R2: {r2_kr:.4f}")

### 4.2. Bayesian Linear Regression
- Dự đoán khoảng tin cậy của biến mục tiêu.

In [ ]:
from part2.advanced_methods import bayesian_regression
y_pred_bay, m_n, S_n, y_var = bayesian_regression(X_train, y_train, X_test)
mae_bay, rmse_bay, r2_bay = compute_metrics(y_test, y_pred_bay)
print(f"Bayesian Linear Regression -> MAE: {mae_bay:.4f}, RMSE: {rmse_bay:.4f}, R2: {r2_bay:.4f}")